In [ ]:
!pip install -q sentence-transformers requests

In [ ]:
import os, re, json, time, torch, requests
from sentence_transformers import SentenceTransformer, util

In [ ]:

from google.colab import files
uploaded = files.upload()

os.makedirs('schema_cache', exist_ok=True)
CACHE_FILES = ('emb_prop.pt', 'metadata_prop.json')
for fname in uploaded:
    if fname in CACHE_FILES:
        os.rename(fname, os.path.join('schema_cache', fname))
        print(f'Moved {fname} → schema_cache/{fname}')



Saving metadata_prop.json to metadata_prop.json
Saving emb_prop.pt to emb_prop.pt
Moved metadata_prop.json → schema_cache/metadata_prop.json
Moved emb_prop.pt → schema_cache/emb_prop.pt


## Instance Retriever (Wikidata API)

In [ ]:

def get_wikidata_instance(entity_string):

    url  = "https://www.wikidata.org/w/api.php"
    hdrs = {"user-agent": "TextToSparqlBot/1.0"}

    fuzzy_string = " ".join([word + "~" for word in entity_string.split()])
    search_query = (
        f"{fuzzy_string} haswbstatement:P31 "
        f"-haswbstatement:P279 "
        f"-haswbstatement:P31=Q4167836 "
        f"-haswbstatement:P31=Q4167410"
    )
    params_search = {
        "action": "query", "list": "search",
        "srsearch": search_query, "srnamespace": "0",
        "srlimit": "10", "format": "json"
    }
    try:
        response_search = requests.get(url, params=params_search, headers=hdrs, timeout=10).json()
        results = response_search.get("query", {}).get("search", [])
        for result in results:
            qid = result["title"]
            params_entity = {
                "action": "wbgetentities", "ids": qid,
                "props": "labels|claims", "languages": "en", "format": "json"
            }
            res_data = requests.get(url, params=params_entity, headers=hdrs, timeout=10).json()
            entity   = res_data.get("entities", {}).get(qid, {})
            label    = entity.get("labels", {}).get("en", {}).get("value", "")
            if label:
                junk_prefixes = ("Category:", "Template:", "List of", "Wikimedia ")
                if not label.startswith(junk_prefixes):
                    claims = entity.get("claims", {})
                    if "P31" in claims and "P279" not in claims:
                        return label, qid
    except Exception as e:
        print(f"Connection Error: {e}")
    return None, None



## Schema Retriever (Properties via Embeddings, Classes via API)

In [ ]:

def parse_plan_string(plan_str):
    steps = []
    for i, step_str in enumerate(plan_str.split("||"), start=1):
        step_str = step_str.strip()
        if not step_str: continue
        step_str = re.sub(r"^step\d+:\s*", "", step_str)
        step = {"step": i}
        for part in step_str.split("|"):
            part = part.strip()
            if ":" in part:
                k, v = part.split(":", 1)
                step[k.strip()] = v.strip()
        steps.append(step)
    return steps

def normalise_plan(plan):
    if isinstance(plan, str):  return parse_plan_string(plan)
    if isinstance(plan, dict) and "steps" in plan: return plan["steps"]
    if isinstance(plan, list): return plan
    raise ValueError(f"Unsupported plan type: {type(plan)}")

def _extract_subj_obj(step):
    action    = step.get("action", "")
    join_type = step.get("join_type", "")
    if action == "find_object":
        if join_type == "intersect": return step.get("subject_variable"), step.get("filter_variable")
        return step.get("subject_variable"), step.get("output_variable")
    elif action == "find_subjects": return step.get("output_variable"), step.get("object_variable")
    elif action in ("find_statement", "left_join", "optional_find"): return step.get("subject_variable"), step.get("output_variable")
    elif action == "find_qualifier": return step.get("subject_variable"), step.get("output_variable")
    elif action == "filter_statement": return step.get("filter_variable"), step.get("value_variable") or step.get("value")
    elif action == "verify_fact": return step.get("subject_variable"), step.get("object_variable")
    elif action == "property_path": return step.get("subject_variable"), step.get("object_variable")
    elif action in ("exists", "exclude"): return step.get("filter_variable"), None
    return None, None

def extract_schema_with_variables(plan):
    steps = normalise_plan(plan)
    properties, classes = [], []
    seen_props, seen_classes = set(), set()
    for step in steps:
        action    = step.get("action", "")
        prop_name = step.get("property") or step.get("qualifier_property") or step.get("property_path")
        if prop_name and prop_name not in seen_props:
            seen_props.add(prop_name)
            subj, obj = _extract_subj_obj(step)
            properties.append({"property": prop_name, "subject": subj, "object": obj})
        if action in ("find_by_type", "filter_type") and "type" in step:
            cls = step["type"]
            if cls not in seen_classes:
                seen_classes.add(cls)
                classes.append({"class": cls, "variable": step.get("output_variable") or step.get("filter_variable")})
        if action == "verify_fact" and "object_type" in step:
            cls = step["object_type"]
            if cls not in seen_classes:
                seen_classes.add(cls)
                classes.append({"class": cls, "variable": step.get("object_variable") or step.get("subject_variable")})
    return properties, classes


def get_wikidata_class(class_string, top_k=5):
    url  = "https://www.wikidata.org/w/api.php"
    hdrs = {"user-agent": "TextToSparqlBot/1.0"}
    fuzzy_string  = " ".join([word + "~" for word in class_string.split()])
    params_search = {
        "action": "query", "list": "search",
        "srsearch": fuzzy_string, "srnamespace": "0",
        "srlimit": "10", "format": "json"
    }
    results = []
    try:
        response = requests.get(url, params=params_search, headers=hdrs, timeout=10).json()
        search_results = response.get("query", {}).get("search", [])
        for i, result in enumerate(search_results):
            qid = result["title"]
            params_label = {
                "action": "wbgetentities", "ids": qid,
                "props": "labels|claims", "languages": "en", "format": "json"
            }
            res_entity  = requests.get(url, params=params_label, headers=hdrs, timeout=10).json()
            entity_data = res_entity.get("entities", {}).get(qid, {})
            label       = entity_data.get("labels", {}).get("en", {}).get("value", "")
            if not label: continue
            junk_prefixes = ("Category:", "Template:", "List of", "Wikimedia category")
            if label.startswith(junk_prefixes): continue
            claims   = entity_data.get("claims", {})
            if "P279" in claims:
                results.append({"uri": f"wd:{qid}", "label": label, "score": 1.0 / (i + 1)})
                if len(results) >= top_k: break
        time.sleep(0.15)
    except Exception as e:
        print(f"  API error for '{class_string}': {e}")
    return results


class TextToSPARQLRetriever:
    def __init__(self, cache_dir="schema_cache", model_name="all-MiniLM-L6-v2"):
        print(f"Loading embedding model: {model_name}")
        self.model = SentenceTransformer(model_name)
        print("Loading property index...")
        self.prop_embeddings = torch.load(os.path.join(cache_dir, "emb_prop.pt"), map_location="cpu")
        with open(os.path.join(cache_dir, "metadata_prop.json"), encoding="utf-8") as f:
            self.prop_metadata = json.load(f)
        print(f"  {len(self.prop_metadata)} properties loaded")
        print("  Class retrieval: Wikidata API")


    def _get_top_k_props(self, label, k=10):
        query_emb = self.model.encode(label, convert_to_tensor=True)
        scores    = util.cos_sim(query_emb, self.prop_embeddings)[0]
        top       = torch.topk(scores, k=k)
        results   = []
        for score, idx in zip(top.values, top.indices):
            entry = self.prop_metadata[idx.item()]
            results.append({"uri": entry.get("uri", ""), "label": entry.get("label", ""), "score": score.item()})
        return results

    def process_plan(self, plan, top_k=5):
        plan_props, plan_classes = extract_schema_with_variables(plan)
        prop_raw  = {p["property"]: self._get_top_k_props(p["property"], k=top_k) for p in plan_props}
        class_raw = {}
        for c in plan_classes:
            lbl = c["class"]
            print(f"  Fetching class '{lbl}' from Wikidata API...")
            class_raw[lbl] = get_wikidata_class(lbl, top_k=top_k)
        return {"properties": prop_raw, "classes": class_raw}


retriever = TextToSPARQLRetriever(cache_dir="schema_cache")

Loading embedding model: all-MiniLM-L6-v2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loading property index...
  13348 properties loaded
  Class retrieval: Wikidata API


In [ ]:

def extract_entities_from_plan(plan_str):
    entities = []
    for step_str in plan_str.split("||"):
        if "find_entity" in step_str:
            sf_m  = re.search(r'surface_form:\s*([^|]+)', step_str)
            var_m = re.search(r'output_variable:\s*([^|]+)', step_str)
            if sf_m and var_m:
                entities.append({"surface_form": sf_m.group(1).strip(), "output_variable": var_m.group(1).strip()})
    return entities

def build_transformer_input(question, plan, dataset="Wikidata"):


    entities = extract_entities_from_plan(plan)
    for e in entities:
        label, qid = get_wikidata_instance(e["surface_form"])
        e["retrieved_uri"] = f"wd:{qid}" if qid else e["surface_form"]


    schema_out   = retriever.process_plan(plan, top_k=5)
    prop_results = schema_out["properties"]
    cls_results  = schema_out["classes"]

    retrieved_props = []
    for label, candidates in prop_results.items():
        top_uri = candidates[0]["uri"] if candidates else label
        retrieved_props.append({"label": label, "uri": top_uri})

    retrieved_classes = []
    for label, candidates in cls_results.items():
        top_uri = candidates[0]["uri"] if candidates else label
        retrieved_classes.append({"label": label, "uri": top_uri})


    schema_lines = []
    for e in entities:
        schema_lines.append(f"  {e['surface_form']} -> {e['retrieved_uri']}")
    for p in retrieved_props:
        schema_lines.append(f"  {p['label']} -> {p['uri']}")
    for c in retrieved_classes:
        schema_lines.append(f"  {c['label']} -> {c['uri']}")
    schema_str = "\n".join(schema_lines)


    final_input = (
        f"generate sparql [{dataset}]: {question}\n"
        f"plan: {plan}\n"
        f"schema:\n{schema_str}"
    )
    return final_input



In [ ]:


question = "Which was the wife of Erich Honecker in the series ordinal 3?"

plan = "step1: action: find_entity | surface_form: Erich Honecker | output_variable: ?erichhonecker | semantic_type: ENTITY || step2: action: find_statement | property: spouse | subject_variable: ?erichhonecker | output_variable: ?s | semantic_type: PROPERTY || step3: action: find_object | property: series ordinal | subject_variable: ?s | output_variable: ?x | semantic_type: QUALIFIER | is_qualifier: True || step4: action: find_object | property: spouse | subject_variable: ?s | output_variable: ?obj | semantic_type: PROPERTY || step5: action: filter | filter_variable: ?x | operator: contains | value: 3 | value_type: string | semantic_type: LITERAL"

dataset = "Wikidata"   # "Wikidata" or "DBpedia"


final_input = build_transformer_input(question, plan, dataset)


print(final_input)


generate sparql [Wikidata]: Which was the wife of Erich Honecker in the series ordinal 3?
plan: step1: action: find_entity | surface_form: Erich Honecker | output_variable: ?erichhonecker | semantic_type: ENTITY || step2: action: find_statement | property: spouse | subject_variable: ?erichhonecker | output_variable: ?s | semantic_type: PROPERTY || step3: action: find_object | property: series ordinal | subject_variable: ?s | output_variable: ?x | semantic_type: QUALIFIER | is_qualifier: True || step4: action: find_object | property: spouse | subject_variable: ?s | output_variable: ?obj | semantic_type: PROPERTY || step5: action: filter | filter_variable: ?x | operator: contains | value: 3 | value_type: string | semantic_type: LITERAL
schema:
  Erich Honecker -> wd:Q2607
  spouse -> wd:P26
  series ordinal -> wd:P1545
